In [2]:
# Import Pandas library for data manipulation and analysis
import pandas as pd
# Import urllib module to handle URL parsing, encoding, and web request operations
import urllib
# Import create_engine from SQLAlchemy to establish database connection instances
from sqlalchemy import create_engine

In [6]:
# Load HR attrition dataset from CSV file into a Pandas dataframe
df = pd.read_csv('../data/hr_attrition.csv')

In [7]:
#Convert column names to clean snake_case for SQL Server
df.columns = (
    df.columns.str.strip().str.replace('([a-z0-9])([A-Z])',r'\1_\2',regex=True).str.lower()
)

In [8]:
# Drop redundant or constant columns from the dataframe if they exist
cols_to_drop = ['employee_count', 'over18', 'standard_hours']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

In [9]:
#SQL Server Connection Config
server = 'localhost'
database = 'HR_Analytics'

In [10]:
# Construct SQLAlchemy connection string for SQL Server using Windows authentication
connection_string = (
    f"mssql+pyodbc://{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

In [11]:
# Create a SQLAlchemy engine instance using the constructed connection string
engine = create_engine(connection_string)

In [13]:
#Load Cleaned Data into SQL Server
df.to_sql('hr_emp_attrition',con=engine,if_exists='replace',index=False)
# Log ingestion success message displaying total rows, columns, and target table name
print(f"Successfully cleaned and ingested {len(df)} rows \
with {len(df.columns)} columns into '{database}.dbo.hr_emp_attrition'.")

C:\Users\Hassa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1135.8'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Successfully cleaned and ingested 1470 rows with 32 columns into 'HR_Analytics.dbo.hr_emp_attrition'.


In [14]:
# Create binary flag for SQL rate calculations (1 = Yes, 0 = No)
df['attrition_num'] = df['attrition'].apply(
    lambda x: 1 if str(x).strip().lower()=='yes' else 0
)

In [15]:
# Define label mappings for survey ratings
edu_map = {1: 'Below College', 2: 'College', 3: 'Bachelor', 4: 'Master', 5: 'Doctor'}
satisfaction_map = {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'}
perf_map = {1: 'Low', 2: 'Good', 3: 'Excellent', 4: 'Outstanding'}
wlb_map = {1: 'Bad', 2: 'Good', 3: 'Better', 4: 'Best'}

In [16]:
# Apply mappings
df['education_label'] = df['education'].map(edu_map)
df['env_satisfaction_label'] = df['environment_satisfaction'].map(satisfaction_map)
df['job_satisfaction_label'] = df['job_satisfaction'].map(satisfaction_map)
df['relationship_satisfaction_label'] = df['relationship_satisfaction'].map(satisfaction_map)
df['job_involvement_label'] = df['job_involvement'].map(satisfaction_map)
df['performance_rating_label'] = df['performance_rating'].map(perf_map)
df['work_life_balance_label'] = df['work_life_balance'].map(wlb_map)

In [17]:
# Categorize continuous age and tenure into strategic groups
df['age_group'] = pd.cut(df['age'], bins=[0,25,35,45,55,100], labels=['Under 25', '25-34', '35-44', '45-54', '55+'])
df['tenure_group'] = pd.cut(df['years_at_company'],bins=[-1,2,5,10,100], labels=['0-2 Years', '3-5 Years', '6-10 Years', '10+ Years'])

In [18]:
# Overwrite table with fully cleaned and enriched 42-column dataset
df.to_sql('hr_emp_attrition',con=engine,if_exists='replace',index=False)
print(f"Updated SQL Server: {df.shape[0]} rows and {df.shape[1]} columns successfully saved.")

Updated SQL Server: 1470 rows and 42 columns successfully saved.


In [19]:
# Read back top records directly from SQL Server to verify schema
query = """
select top 5 
    employee_number, 
    department, 
    job_role, 
    attrition, 
    attrition_num, 
    job_satisfaction_label, 
    age_group, 
    tenure_group
from hr_emp_attrition
"""
pd.read_sql(query,con=engine)

,employee_number,department,job_role,attrition,attrition_num,job_satisfaction_label,age_group,tenure_group
0,1,Sales,Sales Executive,Yes,1,Very High,35-44,6-10 Years
1,2,Research & Development,Research Scientist,No,0,Medium,45-54,6-10 Years
2,4,Research & Development,Laboratory Technician,Yes,1,High,35-44,0-2 Years
3,5,Research & Development,Research Scientist,No,0,High,25-34,6-10 Years
4,7,Research & Development,Laboratory Technician,No,0,Medium,25-34,0-2 Years
